In [ ]:
import boto3, botocore
from botocore.exceptions import ClientError
import os, time, json, io, zipfile
from datetime import date
from dotenv import load_dotenv


from misc import load_from_yaml, save_to_yaml
import iam, s3, lf, rds, vpc, ec2

load_dotenv(".env")
# boto3.setup_default_session(profile_name="AMominNJ")

True

In [ ]:
ALL_IN_ONE_SG = 'sg-0d8a868137f653df6'
ACCOUNT_ID        = os.environ['AWS_ACCOUNT_ID_ROOT']
REGION            = os.environ['AWS_DEFAULT_REGION']
VPC_ID            = os.environ['AWS_DEFAULT_VPC']
SECURITY_GROUP_ID = os.environ['AWS_DEFAULT_SG_ID']
SUBNET_IDS        = SUBNET_IDS = os.environ["AWS_DEFAULT_SUBNET_IDS"].split(":")
SUBNET_ID         = SUBNET_IDS[0]
AWS_INSTANCE_ID_JMASTER   = os.environ['AWS_INSTANCE_ID_JMASTER']
AWS_INSTANCE_ID_JAGENT    = os.environ['AWS_INSTANCE_ID_JAGENT']
AWS_DEFAULT_IMAGE_ID      = os.environ['AWS_DEFAULT_IMAGE_ID']
AWS_DEFAULT_KEY_PAIR_NAME = os.environ['AWS_DEFAULT_KEY_PAIR_NAME']
AWS_DEFAULT_INSTANCE_TYPE = os.environ['AWS_DEFAULT_INSTANCE_TYPE']

In [ ]:
sts_client           = boto3.client('sts')
rds_client           = boto3.client('rds')
iam_client           = boto3.client('iam')
s3_client            = boto3.client('s3')
glue_client          = boto3.client('glue')
lakeformation_client = boto3.client('lakeformation')
stepfunctions_client = boto3.client('stepfunctions')
apigateway_client    = boto3.client('apigateway')
lsn_client           = boto3.client('lambda')
events_client        = boto3.client('events')
sqs_client           = boto3.client('sqs')

emr_client = boto3.client('emr', region_name=REGION)

In [ ]:
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)
elbv2_client = boto3.client("elbv2", region_name=REGION)
autoscaling_client = boto3.client("autoscaling", region_name=REGION)
route53_client = boto3.client("route53", region_name=REGION)
acm_client = boto3.client("acm", region_name=REGION)  # Change region as needed

# CloudFormation is a global service, but you can specify a region for the client
cf_client = boto3.client("cloudformation", region_name=REGION)  

# # Example: Get a specific VPC
# vpc = ec2_resource.Vpc('vpc_id')

# # Example: Get a specific EBS volume
# volume = ec2_resource.Volume('volume_id')

-   [CloudFormation Designer Tutorial](https://www.youtube.com/watch?v=AkihqHHWHvs)
-   [Creating a CloudFormation Template for an IAM User](https://www.youtube.com/watch?v=hcEWgwP2xzk)
-   [boto3: CloudFormation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/cloudformation.html)
-   [CloudFormation template snippets](https://docs.aws.amazon.com/AWSCloudFormation/latest/UserGuide/template-snippets.html)
-   [AWS CloudFormation Sample Templates](https://aws.amazon.com/cloudformation/resources/templates/govcloud-us/)

In [ ]:
# Read CloudFormation template from a file
with open("/Users/am/mydocs/Software_Development/Web_Development/aws/boto_scripts/alb-cf-template.yml","r",) as alb_cf_template:
    template_body = alb_cf_template.read()

# Define stack name
stack_name = "securing-alb-with-tls-certificates"
# Define CloudFormation parameters
parameters = [
    {
        "ParameterKey": "VPCId",
        "ParameterValue": os.environ["AWS_DEFAULT_VPC"],
    },
    {
        "ParameterKey": "SubnetIdOne",
        "ParameterValue": os.environ["AWS_DEFAULT_SUBNET_A"],
    },
    {
        "ParameterKey": "SubnetIdTwo",
        "ParameterValue": os.environ["AWS_DEFAULT_SUBNET_C"],
    },
    {
        "ParameterKey": "ImageId",
        "ParameterValue": os.environ["AMAZON_LINUX_AMI_ID"],
    },
    {"ParameterKey": "InstanceType", "ParameterValue": "t2.micro"},
]

# Create CloudFormation stack
print(f"Creating CloudFormation stack: {stack_name}")
response = cf_client.create_stack(
    StackName=stack_name,
    TemplateBody=template_body,
    Parameters=parameters,
    # Capabilities=["CAPABILITY_IAM", "CAPABILITY_NAMED_IAM"],  # Required for IAM resources only if your template creates IAM resources
)

#### Delete Resources

In [ ]:
# Delete the stack
cf_client.delete_stack(StackName=stack_name)